#TASK 1 : Conceptual Understanding
"Love" vs "love": 
In NLP, these are treated as two distinct tokens because of their ASCII values. "Love" (capitalized) might be seen as more significant or the start of a 
sentence, while "love" is the standard lowercase form. Case folding (lowercasing) is usually done to ensure the model treats them as the same semantic concept.

    
Stopwords: 
If not removed, the model may focus on high-frequency but low-value words (like "the", "is", "at"), creating "noise" that can dilute the importance of 
context-heavy words.

    
Harmful Stopword Removal:

Sentiment Analysis: Words like "not" or "no" are often stopwords, but removing them flips the meaning (e.g., "not happy" becomes "happy").

Search Queries/Phases: In a phrase like "To be or not to be", every word is a stopword. Removing them leaves an empty string.

    
Stemming vs Lemmatization:

Stemming: 
A rule-based approach that chops off word endings (e.g., "studies" becomes "studi"). It’s fast but often results in non-dictionary words.

Lemmatization: 
A dictionary-based approach that returns the "root" word or lemma (e.g., "studies" becomes "study") by considering the word's Part-of-Speech (POS).


In [15]:
import re
import pandas as pd
from collections import Counter

In [16]:
#TASK 2 : Build advanced Preprocessing Function
def preprocess_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove URLs and Emails
    text = re.sub(r'http\S+|www\S+|[\w\.-]+@[\w\.-]+', '', text)

    # 3. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. Handle repeated characters (e.g., "soooo" -> "so", "goooood" -> "good")
    # This regex looks for characters repeated 3 or more times and reduces to 1 or 2
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 5. Remove punctuation and special symbols (except spaces)
    text = re.sub(r'[^\w\s]', '', text)

    # 6. Remove extra spaces
    text = " ".join(text.split())

    # 7. Tokenize and remove very short tokens (keep "no", "not")
    keep_words = {'no', 'not'}
    tokens = [word for word in text.split() if len(word) > 2 or word in keep_words]

    return tokens

In [17]:
#Task 6: Build Full Pipeline
def full_pipeline(text_list):
    processed_data = []
    all_tokens = []

    for original in text_list:
        tokens = preprocess_text(original)
        clean_sentence = " ".join(tokens)
        all_tokens.extend(tokens)
        
        # Task 4: Token Analytics
        total_tokens = len(tokens)
        unique_tokens = len(set(tokens))
        avg_len = sum(len(t) for t in tokens) / total_tokens if total_tokens > 0 else 0
        
        processed_data.append({
            "Original Text": original,
            "Cleaned Tokens": tokens,
            "Cleaned Sentence": clean_sentence,
            "Total Tokens": total_tokens,
            "Unique Tokens": unique_tokens,
            "Avg Token Length": round(avg_len, 2)
        })

    return processed_data, all_tokens

In [18]:
#Task 3: Stress Testing
sample_inputs = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results, total_token_list = full_pipeline(sample_inputs)

# Display Results
df = pd.DataFrame(results)
print("--- PROCESSED DATA ---")

#Task 4: Token Analytics
print(df[["Original Text", "Cleaned Sentence", "Total Tokens", "Avg Token Length"]])

--- PROCESSED DATA ---
                        Original Text               Cleaned Sentence  \
0         Get 100% FREE access now!!!            get free access now   
1  I absolutely looooved this product  absolutely loved this product   
2          Worst service ever... 0/10             worst service ever   
3               Call me at 9876543210                           call   
4          This is THE best course!!!           this the best course   
5       Visit https://openai.com now!                      visit now   
6             Nooooo this is baaad!!!                    no this bad   
7                   OK OK OK I got it                            got   
8     Win $$$ now!!! Limited offer!!!          win now limited offer   
9            I am not happy with this            not happy with this   

   Total Tokens  Avg Token Length  
0             4              4.00  
1             4              6.50  
2             3              5.33  
3             1              4.00  
4   

Task 4: Analysis Questions


Which sentence had the most noise?

The sentence "Win $$$ now!!! Limited offer!!!" 
or "Visit https://openai.com now!" typically 
contains the most noise due to non-alphabetic 
symbols ($) and URLs that provide little 
semantic value for general models.

Which sentence retained the most meaningful tokens?

"I absolutely looooved this product" usually 
retains the most meaning because even after 
normalization ("looooved" to "loved"), the core 
sentiment-carrying words remain intact.

In [22]:
#Task 5: Frequency Analysis
word_freq = Counter(total_token_list)
print("\n--- TOP 10 MOST FREQUENT WORDS ---")
print(word_freq.most_common(10))

print("\n--- TOP 5 LEAST FREQUENT WORDS ---")
print(word_freq.most_common()[:-6:-1])


--- TOP 10 MOST FREQUENT WORDS ---
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

--- TOP 5 LEAST FREQUENT WORDS ---
[('with', 1), ('happy', 1), ('not', 1), ('offer', 1), ('limited', 1)]


In [23]:
#Task 7: Error Handling
print("\n--- ERROR HANDLING CHECKS ---")
error_cases = ["", "😊😊😊", "123456"]
for case in error_cases:
    print(f"Input: '{case}' -> Output: {preprocess_text(case)}")


--- ERROR HANDLING CHECKS ---
Input: '' -> Output: 
Input: '😊😊😊' -> Output: []
Input: '123456' -> Output: []
